# Exploración del catálogo StreamView Analytics

Este notebook utiliza el proceso reproducible del proyecto y las mismas tablas destinadas a Power BI. Ejecutar las celdas en orden. No contiene un dashboard alternativo.


In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import Image, display

root = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "src/main.py").exists())
for script in ["main.py", "referencias_kpis.py", "analisis_catalogo.py"]:
    subprocess.run([sys.executable, str(root / "src" / script)], check=True)
data = root / "data/processed"
catalog = pd.read_csv(data / "catalogo.csv")


## Identidad y calidad

Una clave combina tipo e identificador. Se conservan ambos tipos cuando comparten `show_id`. Las excepciones se documentan en archivos separados; los nulos financieros no se imputan.


In [ ]:
display(pd.read_csv(data / "calidad_columnas.csv"))
display(pd.read_csv(data / "referencia_kpis.csv").head(3))
assert catalog.content_id.is_unique
assert len(catalog) == 31991


## Composición y concentración

Los conteos de géneros y países corresponden a asociaciones. Un título puede pertenecer a varias categorías, por lo que la suma de categorías puede superar el total.


In [ ]:
for name in ["tipos", "generos", "paises", "idiomas"]:
    display(pd.read_csv(data / f"resumen_{name}.csv").head(10))
for name in ["composicion_catalogo", "generos_catalogo", "paises_catalogo", "idiomas_catalogo"]:
    display(Image(filename=str(root / "images" / f"{name}.png")))


## Distribución temporal y popularidad

La distribución temporal de la muestra es uniforme; no permite inferir crecimiento operativo. La popularidad es un índice relativo, no reproducciones.


In [ ]:
display(Image(filename=str(root / "images/estrenos_catalogo.png")))
display(Image(filename=str(root / "images/popularidad_generos.png")))


## Finanzas y alcance

La razón ingresos/presupuesto no representa beneficio neto ni rentabilidad de StreamView. Las alertas de presupuesto son provisionales. No hay datos para medir usuarios, retención o consumo real.


In [ ]:
financial = catalog[catalog.has_valid_financials]
print(f"Cobertura financiera: {len(financial)} / {catalog.type.eq("Movie").sum()} películas")
display(pd.read_csv(data / "presupuestos_revision.csv")[["content_id", "title", "budget", "revenue", "revenue_budget_ratio"]])
assert financial.type.eq("Movie").all()
